# Validation

In [35]:
from pathlib import Path
import json
import re
import sys
from rdflib import Graph, Namespace
from rdflib.namespace import DCTERMS

def find_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "ontology").exists():
            return candidate
    return current

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.parsers.huggingFace_new_parser import parse

JSON_PATH = ROOT / "mapping" / "input" / "datasets" / "datasets.json"
TTL_DIRS = [ROOT / "mapping" / "output" / "datasets" ]

MLUO = Namespace("http://example.org/mluo/ontology#")
MLUO_TH = Namespace("http://example.org/mluo/thesaurus#")
DATA_NS = Namespace("http://example.org/mluo/data#")

print({"root": str(ROOT), "json_exists": JSON_PATH.exists(), "ttl_dirs": [str(d) for d in TTL_DIRS]})

{'root': 'C:\\Users\\sunburst-user\\Documents\\datalens', 'json_exists': True, 'ttl_dirs': ['C:\\Users\\sunburst-user\\Documents\\datalens\\mapping\\output\\datasets']}


In [36]:
def norm(x):
    return re.sub(r"[^a-z0-9]+", "-", str(x).strip().lower()).strip("-")

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

json_rows = [obj for obj in raw if isinstance(obj, dict)]
parsed = [parse(obj) for obj in json_rows]
print({"json_raw": len(raw), "json_rows": len(json_rows), "json_parsed": len(parsed)})

{'json_raw': 10000, 'json_rows': 10000, 'json_parsed': 10000}


In [37]:
def json_by_dataset(parsed_rows):
    by_dataset = {}

    for row in parsed_rows:
        ds = row.get("dct_identifier")
        if not ds:
            continue

        usage = row.get("hasUsage", {}) or {}
        tasks_set = {norm(t) for t in as_list(usage.get("hasTask")) if norm(t)}
        cats_set = {norm(c) for c in as_list(usage.get("hasTaskCategory")) if norm(c)}

        by_dataset[ds] = {
            "tasks_set": tasks_set,
            "categories_set": cats_set,
        }

    return by_dataset

json_by_ds = json_by_dataset(parsed)
print({"json_datasets": len(json_by_ds)})

{'json_datasets': 10000}


In [38]:
def load_ttl_graph(ttl_dirs):
    graph = Graph()
    for directory in ttl_dirs:
        for ttl_file in sorted(directory.glob("*.ttl")):
            graph.parse(ttl_file, format="turtle")
    return graph

g = load_ttl_graph(TTL_DIRS)
print({"ttl_triples": len(g)})

{'ttl_triples': 330551}


In [39]:
def canonical(uri):
    s = str(uri)
    if s.startswith(str(MLUO_TH)):
        return norm(s.split("#", 1)[-1])
    if s.startswith(str(DATA_NS) + "task/"):
        return norm(s.split("/task/", 1)[-1].replace("_", "-"))
    if s.startswith(str(DATA_NS) + "task_category/"):
        return norm(s.split("/task_category/", 1)[-1].replace("_", "-"))
    return norm(s)

print("canonical() prête")

canonical() prête


In [40]:
# ============================================================================
# CONTRÔLE GLOBAL - 1/3: construire la vue TTL par dataset
# ============================================================================

ttl_by_ds = {}
for dataset_uri, dataset_id in g.subject_objects(DCTERMS.identifier):
    ds = str(dataset_id)
    tasks_set = set()
    cats_set = set()

    for usage in g.objects(dataset_uri, MLUO.hasUsage):
        for task in g.objects(usage, MLUO.hasTask):
            task_name = canonical(task)
            if task_name:
                tasks_set.add(task_name)
        for cat in g.objects(usage, MLUO.hasTaskCategory):
            cat_name = canonical(cat)
            if cat_name:
                cats_set.add(cat_name)

    ttl_by_ds[ds] = {
        "tasks_set": tasks_set,
        "categories_set": cats_set,
    }

print({"ttl_datasets": len(ttl_by_ds)})

{'ttl_datasets': 10000}


In [41]:
# ============================================================================
# CONTRÔLE GLOBAL - 2/3: calcul des stats et écarts JSON↔TTL
# ============================================================================

json_dataset_set = set(json_by_ds.keys())
json_task_set = set().union(*(v["tasks_set"] for v in json_by_ds.values())) if json_by_ds else set()
json_cat_set = set().union(*(v["categories_set"] for v in json_by_ds.values())) if json_by_ds else set()

ttl_dataset_set = set(ttl_by_ds.keys())
ttl_task_set = set().union(*(v["tasks_set"] for v in ttl_by_ds.values())) if ttl_by_ds else set()
ttl_cat_set = set().union(*(v["categories_set"] for v in ttl_by_ds.values())) if ttl_by_ds else set()

missing_datasets_in_ttl = sorted(json_dataset_set - ttl_dataset_set)
extra_datasets_in_ttl = sorted(ttl_dataset_set - json_dataset_set)
missing_tasks_in_ttl = sorted(json_task_set - ttl_task_set)
extra_tasks_in_ttl = sorted(ttl_task_set - json_task_set)
missing_cats_in_ttl = sorted(json_cat_set - ttl_cat_set)
extra_cats_in_ttl = sorted(ttl_cat_set - json_cat_set)

global_quick_ok = (
    len(missing_datasets_in_ttl) == 0
    and len(missing_tasks_in_ttl) == 0
    and len(missing_cats_in_ttl) == 0
)

print("stats globales calculées")

stats globales calculées


In [42]:
# ============================================================================
# CONTRÔLE GLOBAL - 3/3: affichage rapide (go / no-go)
# ============================================================================

print("=" * 70)
print("CONTRÔLE GLOBAL JSON vs TTL")
print("=" * 70)
print(f"Datasets JSON / TTL          : {len(json_dataset_set)} / {len(ttl_dataset_set)}")
print(f"Tasks uniques JSON / TTL     : {len(json_task_set)} / {len(ttl_task_set)}")
print(f"Categories uniques JSON / TTL: {len(json_cat_set)} / {len(ttl_cat_set)}")

print("\nÉcarts globaux (détection rapide):")
print(f"- Datasets JSON manquants en TTL : {len(missing_datasets_in_ttl)}")
print(f"- Tasks JSON manquantes en TTL   : {len(missing_tasks_in_ttl)}")
print(f"- Cats JSON manquantes en TTL    : {len(missing_cats_in_ttl)}")
print(f"- Datasets en trop dans TTL      : {len(extra_datasets_in_ttl)}")
print(f"- Tasks en trop dans TTL         : {len(extra_tasks_in_ttl)}")
print(f"- Cats en trop dans TTL          : {len(extra_cats_in_ttl)}")

if global_quick_ok:
    print("\n✅ Contrôle global OK: pas de perte détectée côté JSON→TTL.")
else:
    print("\n⚠️ Alerte globale: pertes possibles détectées, lancer/analyser la comparaison détaillée.")

print("\nExemples (max 5) pour orienter le debug:")
print(f"- Datasets manquants TTL: {missing_datasets_in_ttl[:5]}")
print(f"- Tasks manquantes TTL  : {missing_tasks_in_ttl[:5]}")
print(f"- Cats manquantes TTL   : {missing_cats_in_ttl[:5]}")

CONTRÔLE GLOBAL JSON vs TTL
Datasets JSON / TTL          : 10000 / 10000
Tasks uniques JSON / TTL     : 67 / 67
Categories uniques JSON / TTL: 55 / 55

Écarts globaux (détection rapide):
- Datasets JSON manquants en TTL : 129
- Tasks JSON manquantes en TTL   : 0
- Cats JSON manquantes en TTL    : 0
- Datasets en trop dans TTL      : 129
- Tasks en trop dans TTL         : 0
- Cats en trop dans TTL          : 0

⚠️ Alerte globale: pertes possibles détectées, lancer/analyser la comparaison détaillée.

Exemples (max 5) pour orienter le debug:
- Datasets manquants TTL: ['GEM-submissions/GEM__bart_base_schema_guided_dialog__1645547915', 'GEM-submissions/Leo__bart-large__1645784880', 'GEM-submissions/Leo__mbart-large-cc25__1645802644', 'GEM-submissions/lewtun__hugging-face-test-t5-base.outputs.json-36bf2a59__1645558682', 'GEM-submissions/lewtun__hugging-face-test-t5-base.outputs.json-36bf2a59__1645559101']
- Tasks manquantes TTL  : []
- Cats manquantes TTL   : []


# Contrôle global puis comparaison détaillée JSON → TTL + validation SHACL
1) **Contrôle statistique global** : vérifier rapidement qu’aucun dataset/concept n’a été perdu.
2) **Comparaison détaillée** : analyser les écarts sur un échantillon JSON (triplets importants).
3) **Validation SHACL** : vérifier la conformité RDF globale.

In [43]:
# ============================================================================
# COMPARAISON DÉTAILLÉE - 1/3: paramètres + échantillon JSON
# ============================================================================

import math
import random

SAMPLE_PERCENTAGE = 10.0   # % de datasets JSON à comparer
RANDOM_SEED = 42           # reproductibilité
MAX_PRINT_PER_SECTION = 5  # limite d'affichage

json_dataset_ids = sorted(json_by_ds.keys())
if not json_dataset_ids:
    raise ValueError("Aucun dataset trouvé côté JSON.")

sample_ratio = max(0.0, min(100.0, float(SAMPLE_PERCENTAGE))) / 100.0
sample_size = max(1, math.ceil(len(json_dataset_ids) * sample_ratio))
sample_size = min(sample_size, len(json_dataset_ids))

rng = random.Random(RANDOM_SEED)
selected_dataset_ids = sorted(rng.sample(json_dataset_ids, sample_size))

q_ttl_important_template = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX mluo: <http://example.org/mluo/ontology#>

SELECT ?id ?task ?cat WHERE {
  ?ds dct:identifier ?id .
  FILTER(?id = __DATASET_ID__)
  OPTIONAL {
    ?ds mluo:hasUsage ?u_task .
    ?u_task mluo:hasTask ?task .
  }
  OPTIONAL {
    ?ds mluo:hasUsage ?u_cat .
    ?u_cat mluo:hasTaskCategory ?cat .
  }
}
"""

print("=" * 70)
print("COMPARAISON JSON vs TTL (TRIPLETS IMPORTANTS)")
print("=" * 70)
print(f"Datasets JSON totaux  : {len(json_dataset_ids)}")
print(f"Pourcentage demandé   : {sample_ratio*100:.2f}%")
print(f"Datasets sélectionnés : {len(selected_dataset_ids)}")

COMPARAISON JSON vs TTL (TRIPLETS IMPORTANTS)
Datasets JSON totaux  : 10000
Pourcentage demandé   : 10.00%
Datasets sélectionnés : 1000


In [44]:
# ============================================================================
# COMPARAISON DÉTAILLÉE - 2/3: calcul des écarts sur l'échantillon
# ============================================================================

dataset_reports = []
all_missing_in_ttl = []
all_extra_in_ttl = []
missing_in_ttl_graph_counter = 0

for dataset_id in selected_dataset_ids:
    json_entry = json_by_ds[dataset_id]

    json_important_triples = {("dataset", "dct:identifier", dataset_id)}
    for task in sorted(json_entry.get("tasks_set", set())):
        json_important_triples.add(("usage", "mluo:hasTask", task))
    for cat in sorted(json_entry.get("categories_set", set())):
        json_important_triples.add(("usage", "mluo:hasTaskCategory", cat))

    ttl_entry = ttl_by_ds.get(dataset_id)

    ttl_important_triples = set()
    if ttl_entry is None:
        missing_in_ttl_graph_counter += 1
    else:
        ttl_important_triples.add(("dataset", "dct:identifier", dataset_id))
        for task in sorted(ttl_entry.get("tasks_set", set())):
            ttl_important_triples.add(("usage", "mluo:hasTask", task))
        for cat in sorted(ttl_entry.get("categories_set", set())):
            ttl_important_triples.add(("usage", "mluo:hasTaskCategory", cat))

    missing_in_ttl = sorted(json_important_triples - ttl_important_triples)
    extra_in_ttl = sorted(ttl_important_triples - json_important_triples)
    common_triples = sorted(json_important_triples & ttl_important_triples)

    dataset_reports.append({
        "dataset": dataset_id,
        "status": "ok" if (not missing_in_ttl and not extra_in_ttl) else "diff",
        "json_triples": len(json_important_triples),
        "ttl_triples": len(ttl_important_triples),
        "common": len(common_triples),
        "missing_in_ttl": missing_in_ttl,
        "extra_in_ttl": extra_in_ttl,
    })

    for t in missing_in_ttl:
        all_missing_in_ttl.append((dataset_id, t))
    for t in extra_in_ttl:
        all_extra_in_ttl.append((dataset_id, t))

identical_count = sum(1 for r in dataset_reports if r["status"] == "ok")
diff_count = sum(1 for r in dataset_reports if r["status"] == "diff")
sample_all_identical = diff_count == 0 and missing_in_ttl_graph_counter == 0

print("comparaison détaillée calculée")

comparaison détaillée calculée


In [45]:
# ============================================================================
# COMPARAISON DÉTAILLÉE - 3/3: affichage des résultats
# ============================================================================

print("\n" + "=" * 70)
print("SYNTHÈSE ÉCHANTILLON JSON")
print("=" * 70)
print(f"Datasets identiques                 : {identical_count}")
print(f"Datasets avec différences           : {diff_count}")
print(f"Datasets JSON absents du TTL        : {missing_in_ttl_graph_counter}")
print(f"Triplets attendus mais absents TTL  : {len(all_missing_in_ttl)}")
print(f"Triplets en trop dans TTL           : {len(all_extra_in_ttl)}")

if diff_count > 0:
    print("\nExemples de datasets avec différences:")
    shown = 0
    for r in dataset_reports:
        if r["status"] == "diff":
            print(f"  - {r['dataset']}: JSON={r['json_triples']} | TTL={r['ttl_triples']} | communs={r['common']}")
            shown += 1
            if shown >= MAX_PRINT_PER_SECTION:
                break

print("\n❌ Triplets attendus (JSON) mais absents en TTL - exemples:")
if all_missing_in_ttl:
    for ds_id, t in all_missing_in_ttl[:MAX_PRINT_PER_SECTION]:
        print(f"  - {ds_id}: {t}")
else:
    print("  Aucun")

print("\n➕ Triplets présents en TTL mais absents du JSON - exemples:")
if all_extra_in_ttl:
    for ds_id, t in all_extra_in_ttl[:MAX_PRINT_PER_SECTION]:
        print(f"  - {ds_id}: {t}")
else:
    print("  Aucun")

print("\n" + "=" * 70)
print("VERDICT ÉCHANTILLON")
print("=" * 70)
print("✅ Tous les datasets JSON échantillonnés sont identiques au TTL" if sample_all_identical else "⚠️ Des différences existent entre JSON et TTL sur l'échantillon")


SYNTHÈSE ÉCHANTILLON JSON
Datasets identiques                 : 984
Datasets avec différences           : 16
Datasets JSON absents du TTL        : 16
Triplets attendus mais absents TTL  : 16
Triplets en trop dans TTL           : 0

Exemples de datasets avec différences:
  - GEM-submissions/Leo__bart-large__1645784880: JSON=1 | TTL=0 | communs=0
  - GEM-submissions/lewtun__this-is-a-test-name__1648111972: JSON=1 | TTL=0 | communs=0
  - Gopher-Lab/huberman_lab_Dr._David_Spiegel_Using_Hypnosis_to_Enhance_Mental__Physical_Health__Performance: JSON=1 | TTL=0 | communs=0
  - autoevaluate/autoeval-staging-eval-project-Blaise-g__SumPubmed-3c512f6e-12265640: JSON=1 | TTL=0 | communs=0
  - autoevaluate/autoeval-staging-eval-project-Blaise-g__SumPubmed-54a73f7a-12235635: JSON=1 | TTL=0 | communs=0

❌ Triplets attendus (JSON) mais absents en TTL - exemples:
  - GEM-submissions/Leo__bart-large__1645784880: ('dataset', 'dct:identifier', 'GEM-submissions/Leo__bart-large__1645784880')
  - GEM-submiss

In [46]:
# ============================================================================
# VALIDATION SHACL - 1/3: préparation + exécution
# ============================================================================

print("\n" + "="*70)
print("VALIDATION SHACL - CONFORMITÉ DES DONNÉES")
print("="*70)

violations = []
warnings = []
info_msgs = []
conforms = False
SHACL_AVAILABLE = False

try:
    from pyshacl import validate
    from rdflib.namespace import SH, RDF
    SHACL_AVAILABLE = True
    print("\n✓ pyshacl est disponible")
except ImportError:
    print("\n✗ pyshacl n'est pas installé")
    print("  Installation: pip install pyshacl")

if SHACL_AVAILABLE:
    SHACL_FILE = ROOT / "case-study" / "test" / "shacl-constraints.ttl"
    if SHACL_FILE.exists():
        print(f"✓ Fichier SHACL: {SHACL_FILE.name}")
        shacl_graph = Graph()
        shacl_graph.parse(SHACL_FILE, format="turtle")
        conforms, results_graph, results_text = validate(g, shacl_graph=shacl_graph)
    else:
        print(f"\n✗ Fichier SHACL non trouvé: {SHACL_FILE}")
        SHACL_AVAILABLE = False


VALIDATION SHACL - CONFORMITÉ DES DONNÉES

✗ pyshacl n'est pas installé
  Installation: pip install pyshacl


In [47]:
# ============================================================================
# VALIDATION SHACL - 2/3: lecture des résultats
# ============================================================================

if SHACL_AVAILABLE and 'results_graph' in globals():
    print(f"\n{'='*70}")
    if conforms:
        print("✅ VALIDATION RÉUSSIE")
        print("Les données sont conformes aux contraintes SHACL")
    else:
        print("❌ VIOLATIONS DÉTECTÉES")
    print(f"{'='*70}")

    for validation_result in results_graph.subjects(RDF.type, SH.ValidationResult):
        severity_list = list(results_graph.objects(validation_result, SH.severity))
        message_list = list(results_graph.objects(validation_result, SH.message))
        focus_list = list(results_graph.objects(validation_result, SH.focusNode))

        severity = str(severity_list[0]) if severity_list else "Unknown"
        message = str(message_list[0]) if message_list else "No message"
        focus = str(focus_list[0]) if focus_list else "Unknown"

        result_item = {"severity": severity, "message": message, "node": focus}

        if "Violation" in severity:
            violations.append(result_item)
        elif "Warning" in severity:
            warnings.append(result_item)
        else:
            info_msgs.append(result_item)

    print(f"Violations: {len(violations)} | Warnings: {len(warnings)} | Infos: {len(info_msgs)}")

In [48]:
# ============================================================================
# VALIDATION SHACL - 3/3: exemples lisibles
# ============================================================================

if SHACL_AVAILABLE and 'results_graph' in globals():
    if violations:
        print(f"\n🔴 VIOLATIONS ({len(violations)}):")
        for v in violations[:10]:
            print(f"  • {v['message']}")
        if len(violations) > 10:
            print(f"  ... et {len(violations) - 10} autres violations")

    if warnings:
        print(f"\n🟡 AVERTISSEMENTS ({len(warnings)}):")
        for w in warnings[:10]:
            print(f"  • {w['message']}")
        if len(warnings) > 10:
            print(f"  ... et {len(warnings) - 10} autres avertissements")

    if info_msgs:
        print(f"\n🔵 INFOS ({len(info_msgs)}):")
        for msg in info_msgs[:5]:
            print(f"  • {msg['message']}")
        if len(info_msgs) > 5:
            print(f"  ... et {len(info_msgs) - 5} autres informations")

In [49]:
# ============================================================================
# RÉSUMÉ FINAL (lisible humain)
# ============================================================================

print("\n" + "=" * 70)
print("RÉSUMÉ FINAL")
print("=" * 70)

print("Contrôle global JSON→TTL:")
print(f"- Datasets JSON / TTL          : {len(json_dataset_set)} / {len(ttl_dataset_set)}")
print(f"- Tasks uniques JSON / TTL     : {len(json_task_set)} / {len(ttl_task_set)}")
print(f"- Categories uniques JSON / TTL: {len(json_cat_set)} / {len(ttl_cat_set)}")
print(f"- Datasets manquants en TTL    : {len(missing_datasets_in_ttl)}")
print(f"- Tasks manquantes en TTL      : {len(missing_tasks_in_ttl)}")
print(f"- Cats manquantes en TTL       : {len(missing_cats_in_ttl)}")

print("\nComparaison détaillée (échantillon):")
print(f"- Échantillon JSON analysé : {len(selected_dataset_ids)} datasets ({sample_ratio*100:.2f}%)")
print(f"- Datasets identiques      : {identical_count}")
print(f"- Datasets avec écarts     : {diff_count}")
print(f"- JSON absents du TTL      : {missing_in_ttl_graph_counter}")
print(f"- Triplets manquants TTL   : {len(all_missing_in_ttl)}")
print(f"- Triplets en trop TTL     : {len(all_extra_in_ttl)}")

if SHACL_AVAILABLE:
    print("\nValidation SHACL:")
    print(f"- Conforme        : {conforms}")
    print(f"- Violations      : {len(violations)}")
    print(f"- Avertissements  : {len(warnings)}")
else:
    print("\nValidation SHACL: non exécutée (pyshacl absent ou fichier manquant)")

if global_quick_ok and sample_all_identical and SHACL_AVAILABLE and conforms:
    print("\n✅ Verdict: global OK, détail OK, SHACL conforme.")
elif global_quick_ok and sample_all_identical:
    print("\n⚠️ Verdict: global/détail OK, mais SHACL a des points à corriger.")
else:
    print("\n⚠️ Verdict: problème détecté au global ou au détail (voir écarts affichés).")


RÉSUMÉ FINAL
Contrôle global JSON→TTL:
- Datasets JSON / TTL          : 10000 / 10000
- Tasks uniques JSON / TTL     : 67 / 67
- Categories uniques JSON / TTL: 55 / 55
- Datasets manquants en TTL    : 129
- Tasks manquantes en TTL      : 0
- Cats manquantes en TTL       : 0

Comparaison détaillée (échantillon):
- Échantillon JSON analysé : 1000 datasets (10.00%)
- Datasets identiques      : 984
- Datasets avec écarts     : 16
- JSON absents du TTL      : 16
- Triplets manquants TTL   : 16
- Triplets en trop TTL     : 0

Validation SHACL: non exécutée (pyshacl absent ou fichier manquant)

⚠️ Verdict: problème détecté au global ou au détail (voir écarts affichés).
